# Подготовка

In [1]:
import os
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')

from aee.config import settings
from aee.tasks import TASK_REGISTRY
from aee.utils import load_ground_truth, load_predictions
from aee.evaluation.matcher import ExperimentMatcher
from aee.tasks.nanozymes import NanozymeExperiment

TASK_NAME = "nanozymes"
TASK_CONF = TASK_REGISTRY[TASK_NAME]
COMPARE_FIELDS = TASK_CONF["compare_fields"]

matcher = ExperimentMatcher(fields_to_compare=COMPARE_FIELDS)

print(f"Корневая директория: {Path.cwd()}")
print(f"Поля для сравнения: {COMPARE_FIELDS}")

Корневая директория: /home/arutamonofu/projects/autoevoextractor
Поля для сравнения: ['formula', 'activity', 'reaction_type', 'ph', 'temperature', 'surface', 'syngony', 'length', 'width', 'depth', 'km_value', 'vmax_value', 'c_min', 'c_max', 'c_const', 'ccat_value', 'km_unit', 'vmax_unit', 'c_const_unit', 'ccat_unit']


In [2]:
gt_path = settings.paths.ground_truth_dir / f"{TASK_NAME}.csv"
gt_dict = load_ground_truth(gt_path, TASK_CONF["row_converter"])

path_manual = Path("data/predictions/manual")
preds_manual = load_predictions(path_manual, NanozymeExperiment)

path_auto = Path("data/predictions/auto")
preds_auto = load_predictions(path_auto, NanozymeExperiment)

path_zero = Path("data/predictions/zeroshot")
preds_zero = load_predictions(path_auto, NanozymeExperiment)

print(f"Загружено GT: {len(gt_dict)}")
print(f"Загружено Manual: {len(preds_manual)}")
print(f"Загружено Auto: {len(preds_auto)}")
print(f"Загружено Zero-shot: {len(preds_auto)}")

sample_id = list(preds_manual.keys())[0]
print(f"\nПример ID: {sample_id}")
print(f"Экспериментов в GT: {len(gt_dict.get(sample_id, []))}")

Загружено GT: 395
Загружено Manual: 18
Загружено Auto: 19
Загружено Zero-shot: 19

Пример ID: j.msec.2018.10.038
Экспериментов в GT: 1


# Расчёт метрик

In [3]:
all_doc_results = []

agents_to_compare = {
    "manual": preds_manual,
    "auto": preds_auto,
    "zero": preds_zero
}

for agent_name, preds_dict in agents_to_compare.items():
    for doc_id, preds in preds_dict.items():
        gts = gt_dict.get(doc_id, [])
        
        pairs = matcher.align_pairs(preds, gts)
        stats = matcher._compute_stats(pairs)
        
        all_doc_results.append({
            "agent": agent_name,
            "doc_id": doc_id,
            "f1": stats["f1"],
            "precision": stats["precision"],
            "recall": stats["recall"],
            "n_pred": len(preds),
            "n_gt": len(gts)
        })

df_docs = pd.DataFrame(all_doc_results)

# Анализ

Результаты ручного агента

In [4]:
df_docs[df_docs['agent'] == 'manual']

,agent,doc_id,f1,precision,recall,n_pred,n_gt
0,manual,j.msec.2018.10.038,0.758621,0.733333,0.785714,1,1
1,manual,j.snb.2017.06.175,0.709677,0.578947,0.916667,2,2
2,manual,j.apsusc.2016.12.067,0.350877,0.555556,0.256410,2,6
3,manual,c4tb00968a,0.540541,0.588235,0.500000,2,4
4,manual,am501830v,0.363636,0.228571,0.888889,7,2
5,manual,j.talanta.2015.03.050,0.647059,0.578947,0.733333,2,2
6,manual,j.jpcs.2021.110534,0.742857,0.684211,0.812500,2,2
7,manual,c8tb01132j,0.000000,0.000000,0.000000,0,2
8,manual,jacs.7b00601,0.250000,0.150000,0.750000,10,2
9,manual,j.microc.2019.104352,0.944444,0.944444,0.944444,2,2


Результаты автоматического агента

In [5]:
df_docs[df_docs['agent'] == 'auto']

,agent,doc_id,f1,precision,recall,n_pred,n_gt
18,auto,j.talanta.2015.03.050,0.750000,0.705882,0.800000,2,2
19,auto,c6qm00149a,0.288770,0.341772,0.250000,13,6
20,auto,c8tb01132j,0.000000,0.000000,0.000000,0,2
21,auto,j.jpcs.2021.110534,0.812500,0.812500,0.812500,2,2
22,auto,acs.analchem.7b02880,0.666667,0.611111,0.733333,2,2
23,auto,j.snb.2021.130266,0.510204,0.454545,0.581395,5,3
24,auto,j.colsurfa.2022.129887,0.689655,0.625000,0.769231,1,1
25,auto,j.msec.2018.10.038,0.857143,0.857143,0.857143,1,1
26,auto,j.microc.2019.104352,0.812500,0.928571,0.722222,2,2
27,auto,c4tb00968a,0.575758,0.730769,0.475000,2,4


In [6]:
df_docs[df_docs['agent'] == 'zero']

,agent,doc_id,f1,precision,recall,n_pred,n_gt
37,zero,j.talanta.2015.03.050,0.750000,0.705882,0.800000,2,2
38,zero,c6qm00149a,0.288770,0.341772,0.250000,13,6
39,zero,c8tb01132j,0.000000,0.000000,0.000000,0,2
40,zero,j.jpcs.2021.110534,0.812500,0.812500,0.812500,2,2
41,zero,acs.analchem.7b02880,0.666667,0.611111,0.733333,2,2
42,zero,j.snb.2021.130266,0.510204,0.454545,0.581395,5,3
43,zero,j.colsurfa.2022.129887,0.689655,0.625000,0.769231,1,1
44,zero,j.msec.2018.10.038,0.857143,0.857143,0.857143,1,1
45,zero,j.microc.2019.104352,0.812500,0.928571,0.722222,2,2
46,zero,c4tb00968a,0.575758,0.730769,0.475000,2,4


Сравнение результатов

In [12]:
df_pivot = df_docs.pivot(index="doc_id", columns="agent", values=["f1", "precision", "recall"])

df_pivot['f1_delta'] = df_pivot[('f1', 'auto')] - df_pivot[('f1', 'manual')]
df_pivot['precision_delta'] = df_pivot[('precision', 'auto')] - df_pivot[('precision', 'manual')]
df_pivot['recall_delta'] = df_pivot[('recall', 'auto')] - df_pivot[('recall', 'manual')]

In [14]:
df_pivot

f1           precision              recall  \
agent                       auto    manual      auto    manual      auto   
doc_id                                                                     
acs.analchem.7b02880    0.666667  0.709677  0.611111  0.687500  0.733333   
acsami.2c20878          0.479452  0.616162  0.603448  0.554545  0.397727   
acsomega.0c00147        0.789474  0.815789  0.789474  0.815789  0.789474   
adfm.202001933          0.733333  0.705882  0.785714  0.666667  0.687500   
am501830v               0.533333  0.363636  0.380952  0.228571  0.888889   
c4tb00968a              0.575758  0.540541  0.730769  0.588235  0.475000   
c5ra11014a              0.620690  0.516129  0.529412  0.421053  0.750000   
c6qm00149a              0.288770  0.464516  0.341772  0.356436  0.250000   
c8tb01132j              0.000000  0.000000  0.000000  0.000000  0.000000   
j.apsusc.2016.12.067    0.327273  0.350877  0.562500  0.555556  0.230769   
j.colsurfa.2022.129887  0.689655  0.465116  0.625000  0.333333  0.769231   
j.jpcs.2021.110534      0.812500  0.742857  0.812500  0.684211  0.812500   
j.microc.2019.104352    0.812500  0.944444  0.928571  0.944444  0.722222   
j.msec.2018.10.038      0.857143  0.758621  0.857143  0.733333  0.857143   
j.snb.2017.06.175       0.647059  0.709677  0.500000  0.578947  0.916667   
j.snb.2021.130266       0.510204       NaN  0.454545       NaN  0.581395   
j.talanta.2015.03.050   0.750000  0.647059  0.705882  0.578947  0.800000   
jacs.7b00601            0.240741  0.250000  0.144444  0.150000  0.722222   
s00604-021-04942-7      0.489796  0.628571  0.705882  0.578947  0.375000   

                                  f1_delta precision_delta recall_delta  
agent                     manual                                         
doc_id                                                                   
acs.analchem.7b02880    0.733333 -0.043011       -0.076389     0.000000  
acsami.2c20878          0.693182 -0.136710        0.048903    -0.295455  
acsomega.0c00147        0.815789 -0.026316       -0.026316    -0.026316  
adfm.202001933          0.750000  0.027451        0.119048    -0.062500  
am501830v               0.888889  0.169697        0.152381     0.000000  
c4tb00968a              0.500000  0.035217        0.142534    -0.025000  
c5ra11014a              0.666667  0.104561        0.108359     0.083333  
c6qm00149a              0.666667 -0.175746       -0.014663    -0.416667  
c8tb01132j              0.000000  0.000000        0.000000     0.000000  
j.apsusc.2016.12.067    0.256410 -0.023604        0.006944    -0.025641  
j.colsurfa.2022.129887  0.769231  0.224539        0.291667     0.000000  
j.jpcs.2021.110534      0.812500  0.069643        0.128289     0.000000  
j.microc.2019.104352    0.944444 -0.131944       -0.015873    -0.222222  
j.msec.2018.10.038      0.785714  0.098522        0.123810     0.071429  
j.snb.2017.06.175       0.916667 -0.062619       -0.078947     0.000000  
j.snb.2021.130266            NaN       NaN             NaN          NaN  
j.talanta.2015.03.050   0.733333  0.102941        0.126935     0.066667  
jacs.7b00601            0.750000 -0.009259       -0.005556    -0.027778  
s00604-021-04942-7      0.687500 -0.138776        0.126935    -0.312500

In [11]:
report = df_docs.dropna().groupby('agent')[['f1', 'precision', 'recall']].mean()
report.loc['auto - manual'] = report.loc['auto'] - report.loc['manual']
report.loc['auto - zero'] = report.loc['auto'] - report.loc['zero']
report.loc['manual - zero'] = report.loc['manual'] - report.loc['zero']
report

,f1,precision,recall
agent,,,
auto,0.569702,0.582585,0.618899
manual,0.568309,0.525362,0.687240
zero,0.569702,0.582585,0.618899
auto - manual,0.001394,0.057223,-0.068342
auto - zero,0.000000,0.000000,0.000000
manual - zero,-0.001394,-0.057223,0.068342
